In [1]:
# import subprocess
# import os

# # Define the base directory where you want to save the files
# base_dir = "sam_dictionaries"

# # Ensure the base directory exists
# os.makedirs(base_dir, exist_ok=True)

# # List of URLs to download
# directories = [f"https://baulab.us/u/smarks/autoencoders/pythia-70m-deduped/attn_out_layer{layer}/10_32768/ae.pt" for layer in range(6)]

# for url in directories:
#     # Extract the layer number from the URL for directory naming
#     layer = url.split("attn_out_layer")[1].split("/")[0]
#     # Define the local directory structure
#     local_dir = os.path.join(base_dir, f"attn_out_layer{layer}", "10_32768")
#     # Ensure the local directory structure exists
#     os.makedirs(local_dir, exist_ok=True)
#     # Run wget command, specifying the current directory to save the file
#     subprocess.run(["wget", "-P", local_dir, "--no-parent", url], check=True)

In [1]:
from nnsight import LanguageModel
from nnsight.alterations import alter, PropertyEnvoy

model = LanguageModel("EleutherAI/pythia-70m-deduped", device_map="auto", dispatch=True)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
import torch
from sam_dictionary import AutoEncoder

ae_list = []

for layer in range(6):
    path = f"./sam_dictionaries/attn_out_layer{layer}/10_32768/ae.pt"

    activation_dim = 512 # dimension of the NN's activations to be autoencoded
    dictionary_size = 64 * activation_dim # number of features in the dictionary
    ae = AutoEncoder(activation_dim, dictionary_size)
    ae.load_state_dict(torch.load(path))
    ae.to("cuda:0")

    ae_list.append(ae)

In [3]:
from nnsight.util import fetch_sub_envoy
from nnsight.envoy import Envoy

target = model.modules(lambda x: x._module_path == ".gpt_neox.layers.0.attention")[0]

# Add envoy to attn
target._add_envoy(ae_list[0], "test")
target._module.__setattr__("test", ae_list[0])

attn = model.gpt_neox.layers[0].attention

with model.trace("trace", scan=False, validate=False):
    activation = attn.output[0]

    attn.output[0][:] = attn.test(activation, traceable=True)

    test = attn.test.encoder.output.save()

You're using a GPTNeoXTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [4]:
test

tensor([[[-0.3076, -0.4710, -0.3119,  ..., -0.3578, -0.3495, -0.4264]]],
       device='cuda:0', grad_fn=<ViewBackward0>)

In [ ]:
name_alterations = {
    "wte" : "embed",
    "wpe" : "pos_embed",
    "drop" : "dropout",
    "h" : "layers" ,
    "ln_f" : "ln_final",
    "lm_head" : "unembed"
}

alter(model, name_alterations=name_alterations)